In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_FACT_PERSON_ROLE_ASSIGNMENTS_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    fact_pra_df = session.sql(f"""
        SELECT
            pra.PRA1_ID,
            pra.POI_POI_ID,
            d.PERSON_ROLE_ASSIGNMENTS_INFO_ID,
            pra.START_DATE,
            pra.END_DATE,
            pra.ADD_TS,
            pra.ADD_USER,
            pra.MOD_TS,
            pra.MOD_USER,
            CURRENT_TIMESTAMP() AS CREATED_DATE,
            pra.DELETED_DATE,
            pra.CHECKSUM
        FROM {STG}.{STG_PERSON_ROLE_ASSIGNMENTS} pra
        LEFT JOIN {DWH}.{DIM_PERSON_ROLE_ASSIGNMENTS_INFO} d
            ON pra.ROLE_TYPE IS NOT DISTINCT FROM d.ROLE_TYPE_AV_ID
           AND pra.ROLE_SUB_TYPE IS NOT DISTINCT FROM d.ROLE_SUB_TYPE_AV_ID
           AND d.UPDATED_DATE IS NULL
    """)

    fact_pra_df.create_or_replace_temp_view("TEMP_FACT_PERSON_ROLE_ASSIGNMENTS")

    merge_result = session.sql(f"""
        MERGE INTO {DWH}.{FACT_PERSON_ROLE_ASSIGNMENTS} tgt
        USING TEMP_FACT_PERSON_ROLE_ASSIGNMENTS src
            ON tgt.PRA1_ID = src.PRA1_ID
        WHEN MATCHED AND (
            src.CHECKSUM <> tgt.CHECKSUM
        ) THEN
            UPDATE SET
                POI_POI_ID = src.POI_POI_ID,
                PERSON_ROLE_ASSIGNMENTS_INFO_ID = src.PERSON_ROLE_ASSIGNMENTS_INFO_ID,
                START_DATE = src.START_DATE,
                END_DATE = src.END_DATE,
                ADD_TS = src.ADD_TS,
                ADD_USER = src.ADD_USER,
                MOD_TS = src.MOD_TS,
                MOD_USER = src.MOD_USER,
                DELETED_DATE = src.DELETED_DATE,
                CHECKSUM = src.CHECKSUM
        WHEN NOT MATCHED THEN
            INSERT (
                PRA1_ID,
                POI_POI_ID,
                PERSON_ROLE_ASSIGNMENTS_INFO_ID,
                START_DATE,
                END_DATE,
                ADD_TS,
                ADD_USER,
                MOD_TS,
                MOD_USER,
                CREATED_DATE,
                DELETED_DATE,
                CHECKSUM
            )
            VALUES (
                src.PRA1_ID,
                src.POI_POI_ID,
                src.PERSON_ROLE_ASSIGNMENTS_INFO_ID,
                src.START_DATE,
                src.END_DATE,
                src.ADD_TS,
                src.ADD_USER,
                src.MOD_TS,
                src.MOD_USER,
                src.CREATED_DATE,
                src.DELETED_DATE,
                src.CHECKSUM
            )
    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_PERSON_ROLE_ASSIGNMENTS load completed. Rows Inserted: {rows_inserted}, Rows Updated: {rows_updated}"
    )

    print(f"DWH LOAD SUCCESS | Rows Inserted: {rows_inserted} | Rows Updated: {rows_updated}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_PERSON_ROLE_ASSIGNMENTS load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")